# Solar Filament Segmentation Challenge 2026: reproducing the final submission

This notebook rebuilds `submission_scale_stable.csv`, the file selected for the final leaderboard
(public PQ 0.44, private PQ 0.42). No training is needed: it loads the two segmentation models stored in `models/`.

**Models.** YOLO26x-seg and YOLOv8x-seg trained at 2048 px on the 707 competition training images by **hdjojo**, published on
Kaggle Models under Apache 2.0 and used unmodified:
[yolo26x-seg-solar-filament](https://www.kaggle.com/models/hdjojo/yolo26x-seg-solar-filament) and
[yolov8x-seg-solar-filament](https://www.kaggle.com/models/hdjojo/yolov8x-seg-solar-filament).

**My contribution: the test-time processing in `filament_tta.py`.**
1. Each model predicts 8 views of every image: the 4 mirror images (original, left-right, up-down, both) at 2048 px and
   the same 4 enlarged to 2560 px, so 16 predictions per image. Every mask is mapped back to the original image.
2. Masks of the same filament from different views (IoU > 0.4, at most one per view) form a group.
3. Each group becomes one mask by confidence-weighted pixel voting (a pixel is kept when it carries at least half of the
   group's confidence).
4. Each filament is ranked by consensus (sum of its confidences / 16) times shape stability (mean IoU between the
   group's masks and the voted mask).
5. Masks are made disjoint in ranking order; a filament that loses more than half of its area is dropped.
6. The 1317 best filaments of the whole test set are kept (1317 is what the two models give at confidence 0.5).

**Run time:** about 95 minutes on one NVIDIA T4 GPU (Kaggle), about 31 s per image. A GPU is strongly recommended.

**Before running:** `pip install -r requirements.txt`, and get the real model files with `git lfs pull` (a plain download
gives 1 KB placeholder files). Put the test images in `data/test/test_images/` or set the environment variable
`TEST_IMAGES_DIR`. On Kaggle, the competition data and the two hdjojo models attached as inputs are found automatically.

In [ ]:
# On Kaggle only ultralytics is missing; elsewhere: pip install -r requirements.txt
import importlib.util, subprocess, sys
if importlib.util.find_spec('ultralytics') is None:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'ultralytics==8.4.214'])

In [ ]:
import glob, os, sys, time
import importlib.metadata as md
import cv2, torch

REPO = os.path.abspath('.')                       # run this notebook from the repository folder
sys.path.insert(0, REPO)
import filament_tta as ft
from ultralytics import YOLO

def version(package):
    try:
        return md.version(package)
    except md.PackageNotFoundError:
        return 'not installed'


print('python', sys.version.split()[0], '|',
      ', '.join(f'{p} {version(p)}' for p in ['ultralytics', 'torch', 'numpy', 'opencv-python', 'pycocotools']))
print('device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU (very slow)')


def find_model(name):
    # the repository's models/ folder first, then Kaggle inputs; Git LFS placeholders (about 1 KB) are skipped
    candidates = [os.path.join(REPO, 'models', name)] + sorted(glob.glob(f'/kaggle/input/**/{name}', recursive=True))
    for p in candidates:
        if os.path.exists(p):
            if os.path.getsize(p) > 1_000_000:
                return p
            print(f'{p} is a Git LFS placeholder: run "git lfs pull" in the repository to get the real file')
    raise FileNotFoundError(f'{name} not found (looked in models/ and /kaggle/input)')


def find_test_dir():
    candidates = [os.environ.get('TEST_IMAGES_DIR', ''), os.path.join(REPO, 'data', 'test', 'test_images')]
    candidates += sorted(glob.glob('/kaggle/input/**/test/test_images', recursive=True))
    for d in candidates:
        if d and glob.glob(os.path.join(d, '*.jpeg')):
            return d
    raise FileNotFoundError('test images not found: put them in data/test/test_images or set TEST_IMAGES_DIR')


TEST_IMAGES_DIR = find_test_dir()
files = sorted(glob.glob(os.path.join(TEST_IMAGES_DIR, '*.jpeg')))
OUT_PATH = os.path.join(REPO, 'outputs', 'submission_scale_stable.csv')
SUBMITTED = os.path.join(REPO, 'submission', 'submission_scale_stable.csv')
print(len(files), 'test images in', TEST_IMAGES_DIR)

In [ ]:
# the two models, with the settings of the original inference notebook (IOU = 0.0: ultralytics NMS setting)
class Config_YOLO26x:
    MODEL_PATH = find_model('yolo26x-seg-solar-filament-2048-1109.pt')
    IMG_SIZE = 2048
    CONF = 0.5
    IOU = 0.0
    MAX_DET = 100


class Config_YOLOv8x:
    MODEL_PATH = find_model('yolov8x-seg-solar-filament-2048-1110.pt')
    IMG_SIZE = 2048
    CONF = 0.5
    IOU = 0.0
    MAX_DET = 100


MODELS = [(YOLO(Config_YOLO26x.MODEL_PATH), Config_YOLO26x), (YOLO(Config_YOLOv8x.MODEL_PATH), Config_YOLOv8x)]
print('models:', Config_YOLO26x.MODEL_PATH, '|', Config_YOLOv8x.MODEL_PATH)

## 1. The method on one image

The 16 views, the groups formed across views, and the filaments kept for this image (before the selection of the
1317 best filaments over the whole test set).

In [ ]:
VIEW_NAMES = {(0, False): 'original', (0, True): 'left-right', (2, True): 'up-down', (2, False): 'both mirrors'}
img = cv2.imread(files[0])
h, w = img.shape[:2]
names, preds = [], []
for size in (2048, ft.BIG):                   # same order as predict_image(): 2048 px views first, then 2560 px
    for (model, cfg), mname in zip(MODELS, ['YOLO26x', 'YOLOv8x']):
        for k, lr in ft.VIEWS:
            names.append(f'{mname} {size} px {VIEW_NAMES[(k, lr)]}')
            preds.append(ft.predict_masks(model, cfg, img, size, k, lr))
print(os.path.basename(files[0]))
for name, p in zip(names, preds):
    print(f'  {name:32s} {0 if p is None else len(p[1]):3d} masks')
cl = ft.clusters_of(preds)
best = sorted(cl, key=lambda c: -c[0] * c[1])[:5]
print(f'{len(cl)} groups; best 5 as (consensus score, stability):', [(round(s, 3), round(st, 3)) for s, st, _ in best])
kept = ft.paint(cl, True, h, w)
print(f'{len(kept)} disjoint filaments kept for this image')
del preds, cl

## 2. All test images → `outputs/submission_scale_stable.csv`

In [ ]:
RESULTS, t0 = {}, time.time()
for n, path in enumerate(files):
    RESULTS[os.path.splitext(os.path.basename(path))[0]] = ft.predict_image(MODELS, cv2.imread(path))
    if (n + 1) % 10 == 0 or n + 1 == len(files):
        el = time.time() - t0
        print(f'{n + 1}/{len(files)} images | {el / (n + 1):.1f} s per image | about '
              f'{el / (n + 1) * (len(files) - n - 1) / 60:.0f} min left', flush=True)
rows, tau = ft.select_top(RESULTS, ft.TARGET)
ft.write_submission(rows, OUT_PATH)
print(f'{len(rows)} filaments (ranking score >= {tau:.3f}) -> {OUT_PATH}')

## 3. Comparison with the file submitted to Kaggle

On the same GPU type the rebuilt file should match the submitted one. On other hardware a few masks can differ by some
pixels (GPU arithmetic is not bit-exact), which does not change the score in a meaningful way.

In [ ]:
if os.path.exists(SUBMITTED):
    res = ft.compare_submissions(OUT_PATH, SUBMITTED)
    print(res)
    if res['different'] == 0 and res['filaments_a'] == res['filaments_b']:
        print('identical to the submitted file')
    else:
        print(f"{res['identical']} identical masks, {res['different']} different "
              f"(mean best IoU with the submitted masks: {res['mean_best_iou_of_different']})")
else:
    print('submission/submission_scale_stable.csv not found: nothing to compare')